# Notebook 05 Part 1: RDD vs DataFrame


In this notebook, we will understand two important Spark data structures:

```text
1. RDD
2. DataFrame
```

---

## Topics Covered

```text
1. What is RDD?
2. What is DataFrame?
3. RDD vs DataFrame
4. Why DataFrame is preferred in modern PySpark
5. When RDD is still useful
6. Create RDD
7. Create DataFrame
8. Convert RDD to DataFrame
9. Basic RDD operations
10. Basic DataFrame operations
11. Summary and practice questions
```

## 1. Simple Meaning

Spark has two important ways to represent data:

| Data Structure | Simple Meaning |
|---|---|
| RDD | Low-level distributed data collection |
| DataFrame | High-level table-like distributed data structure |

Simple line:

```text
RDD is like raw distributed data.
DataFrame is like a distributed table with rows and columns.
```

## 2. What is RDD?

RDD stands for:

```text
Resilient Distributed Dataset
```

| Word | Meaning |
|---|---|

| Resilient | Can recover from failure |
| Distributed | Data is spread across partitions/machines |
| Dataset | Collection of records |

Simple meaning:

```text
RDD is Spark's basic low-level data structure.
It stores distributed data across partitions.
```

RDD gives more control, but it requires more manual coding.

## 3. What is DataFrame?

A DataFrame is a distributed table.

It has:

```text
Rows
Columns
Schema
Data types
```

Example:

| order_id | customer_name | city | amount |
|---|---|---|---|
| 1 | Amit | Delhi | 500 |
| 2 | Priya | Mumbai | 700 |

Simple meaning:

```text
DataFrame is like a table in Spark.
```

DataFrame is preferred in modern PySpark because Spark can optimize it better.

## 4. RDD vs DataFrame

| Point | RDD | DataFrame |
|---|---|---|
| Level | Low-level | High-level |
| Structure | Collection of objects | Table with rows and columns |
| Schema | No fixed schema by default | Has schema |
| Optimization | Less optimized | Optimized by Catalyst optimizer |
| Syntax | Functional style | SQL/table style |
| Ease of use | Harder for beginners | Easier for beginners |
| Performance | Usually slower | Usually faster |
| Best use | Custom low-level logic | Data cleaning, ETL, SQL, analytics |

Simple rule:

```text
Use DataFrame for most PySpark data engineering work.
Use RDD only when low-level control is needed.
```

## 5. Why DataFrame is Preferred

DataFrames are preferred because they support:

```text
1. Column names
2. Schema
3. SQL queries
4. Catalyst optimization
5. Better performance
6. Easier syntax
7. Integration with file formats like CSV, JSON, Parquet
```

In real data engineering projects, we mostly use:

```text
DataFrame + Spark SQL
```

not raw RDDs.

## 6. When RDD is Still Useful

RDDs are less common in modern PySpark, but still useful when:

```text
1. We need very low-level control
2. Data is unstructured
3. Custom processing is difficult with DataFrames
4. We are learning Spark internals
5. We want to understand lineage and transformations deeply
```

But for normal ETL pipelines, DataFrame is the better choice.

## 7. Windows Local Setup Cell

Run this cell before creating `SparkSession`.

This helps avoid the Windows local PySpark error:

```text
Timed out while waiting for the Python worker to connect back
```

In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## 8. Create SparkSession

We will use our Windows-safe SparkSession code.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RDD_vs_DataFrame") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 9. Access SparkContext

RDDs are created using `SparkContext`.

`SparkContext` is available inside `SparkSession`.

In [ ]:
sc = spark.sparkContext

print("Application Name:", sc.appName)
print("Master:", sc.master)
print("Spark Version:", sc.version)

## 10. Create an RDD

We can create an RDD from a Python list using:

```python
sc.parallelize()
```

Simple meaning:

```text
Take local Python data and distribute it as an RDD.
```

In [ ]:
numbers = [1, 2, 3, 4, 5]

rdd = sc.parallelize(numbers)

rdd

## 11. Collect RDD Data

`collect()` brings RDD data back to the driver.

For small data, it is okay.

For large data, avoid `collect()` because it can overload driver memory.

In [ ]:
rdd.collect()

## 12. Basic RDD Transformation: `map()`

`map()` applies a function to every element.

Example:

```text
1 → 10
2 → 20
3 → 30
```

In [ ]:
rdd_multiplied = rdd.map(lambda x: x * 10)

rdd_multiplied.collect()

## 13. Basic RDD Transformation: `filter()`

`filter()` keeps only records that satisfy a condition.

Example: keep numbers greater than 2.

In [ ]:
rdd_filtered = rdd.filter(lambda x: x > 2)

rdd_filtered.collect()

## 14. Create a DataFrame

Now we will create a DataFrame using rows and columns.

In [ ]:
data = [
    (1, "Amit", "Delhi", 500),
    (2, "Priya", "Mumbai", 700),
    (3, "Rahul", "Delhi", 300),
    (4, "Sneha", "Pune", 900)
]

columns = ["order_id", "customer_name", "city", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

## 15. Check DataFrame Schema

DataFrames have schema.

Schema means:

```text
Column names
Data types
Nullable information
```

In [ ]:
df.printSchema()

## 16. Basic DataFrame Operation: `select()`

DataFrame operations are column-based and table-like.

In [ ]:
df.select("customer_name", "amount").show()

## 17. Basic DataFrame Operation: `filter()`

Filtering in DataFrame is easier to read than RDD for table-like data.

In [ ]:
df.filter(df.amount > 500).show()

## 18. Convert RDD to DataFrame

Sometimes we may have RDD data and want to convert it into a DataFrame.

First, create an RDD of tuples.

In [ ]:
rdd_orders = sc.parallelize([
    (1, "Amit", "Delhi", 500),
    (2, "Priya", "Mumbai", 700),
    (3, "Rahul", "Delhi", 300)
])

rdd_orders.collect()

Now convert RDD to DataFrame by providing column names.

In [ ]:
columns = ["order_id", "customer_name", "city", "amount"]

df_from_rdd = rdd_orders.toDF(columns)

df_from_rdd.show()

## 19. Check Schema of DataFrame Created from RDD

Spark automatically infers schema.

In [ ]:
df_from_rdd.printSchema()

## 20. RDD vs DataFrame Code Style

### RDD style

```python
rdd.filter(lambda x: x[3] > 500)
```

Here we use index positions like `x[3]`.

### DataFrame style

```python
df.filter(df.amount > 500)
```

Here we use column names.

DataFrame style is easier to read for structured data.

In [ ]:
# RDD style: amount is at index 3
rdd_orders.filter(lambda x: x[3] > 500).collect()

In [ ]:
# DataFrame style
df_from_rdd.filter(df_from_rdd.amount > 500).show()

## 21. Key Difference: Schema

RDD does not naturally show column names.

DataFrame has column names and data types.

That is why DataFrame is better for:

```text
ETL
Data cleaning
SQL
Reports
Analytics
Data engineering pipelines
```

## 22. Key Difference: Optimization

DataFrames are optimized by Spark's Catalyst optimizer.

Simple meaning:

```text
Spark can understand DataFrame operations better and optimize them.
```

Example:

```python
df.select("city", "amount").filter(df.amount > 500)
```

Spark understands:

```text
Which columns are needed
Which filters can be applied
How to optimize the execution plan
```

RDD gives less optimization because Spark sees it as lower-level Python function logic.

## 23. Which One Should We Use?

For modern PySpark data engineering:

```text
Use DataFrame most of the time.
```

Use RDD only when:

```text
1. You need low-level control
2. You are handling unusual unstructured data
3. You are learning Spark internals
4. DataFrame API cannot easily solve the problem
```

## 24. Final Comparison

| Question | Best Choice |
|---|---|
| Cleaning structured data | DataFrame |
| Reading CSV/JSON/Parquet | DataFrame |
| Running SQL | DataFrame |
| Aggregation and joins | DataFrame |
| Custom low-level processing | RDD |
| Learning Spark internals | RDD |

## 25. Practice Questions

Answer these:

```text
1. What is RDD?
2. What is DataFrame?
3. Why is DataFrame preferred in modern PySpark?
4. What is schema?
5. Which is easier for SQL-like operations: RDD or DataFrame?
6. When should we use RDD?
7. How do we create RDD from a Python list?
8. How do we convert RDD to DataFrame?
```

## 27. Stop SparkSession

At the end of the notebook, stop SparkSession.

In [ ]:
spark.stop()